# SQL Analysis of Support Tickets

Normalising a support ticket dataset into two related tables, then analysing it in SQL.

The dataset stores each ticket's tags as a single text field: "['Account', 'Outage', 'IT']". That's a one-to-many relationship
squashed into one column, so it can't be queried properly as it stands. Splitting it into a separate table makes joins possible.

Data: IT Support Ticket Data (Kaggle), 29,651 tickets.

## 1. Building the database

Two tables:

- `tickets` — one row per ticket, with its department and priority
- `ticket_tags` — one row per ticket per tag, linked by ticket_id

The second table is the main one. Rather than storing five tags in one cell, each becomes its own row, so tags can be counted, grouped and joined against. Indexes are added on both ticket_id and tag, since every query below joins or filters on one of them.

In [5]:
import ast
import sqlite3
import pandas as pd

def q(sql):
    return pd.read_sql_query(sql, conn)
    
CSV = "IT Support Ticket Data.csv"       

df = pd.read_csv(CSV)
df = df[["Body", "Department", "Priority", "Tags"]].dropna(subset=["Department"])
df = df.reset_index(drop=True)
df["ticket_id"] = df.index + 1

# --- tickets table ---
tickets = pd.DataFrame({
    "ticket_id": df["ticket_id"],
    "department": df["Department"],
    "priority": df["Priority"],
    "body_length": df["Body"].astype(str).str.len(),
})

# --- ticket_tags table: one row per ticket per tag ---
rows = []
for tid, raw in zip(df["ticket_id"], df["Tags"]):
    try:
        tags = ast.literal_eval(str(raw))
    except (ValueError, SyntaxError):
        continue
    if not isinstance(tags, list):
        continue
    for tag in tags:
        rows.append({"ticket_id": tid, "tag": str(tag).strip()})

ticket_tags = pd.DataFrame(rows)

conn = sqlite3.connect("tickets.db")
tickets.to_sql("tickets", conn, if_exists="replace", index=False)
ticket_tags.to_sql("ticket_tags", conn, if_exists="replace", index=False)

conn.execute("CREATE INDEX IF NOT EXISTS idx_tags_ticket ON ticket_tags(ticket_id)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_tags_tag ON ticket_tags(tag)")
conn.commit()

print(f"tickets:     {len(tickets):,} rows")
print(f"ticket_tags: {len(ticket_tags):,} rows")
print(f"unique tags: {ticket_tags['tag'].nunique()}")
conn.close()

tickets:     29,651 rows
ticket_tags: 146,390 rows
unique tags: 1650


## Results

29,651 tickets produced 146,390 tag rows which is roughly five tags per ticket. This ratio is why the junction table matters: in the original format, every one of those five tags sat inside a single string.

## 2. Checking the tables

Confirming the split worked before running any analysis.

In [6]:
import sqlite3, pandas as pd
conn = sqlite3.connect("tickets.db")

print(pd.read_sql_query("SELECT * FROM tickets LIMIT 3", conn))
print()
print(pd.read_sql_query("SELECT * FROM ticket_tags LIMIT 8", conn))
print()
print(pd.read_sql_query("""
    SELECT tag, COUNT(*) AS n
    FROM ticket_tags
    GROUP BY tag
    ORDER BY n DESC
    LIMIT 10
""", conn))

   ticket_id             department priority  body_length
0          1      Technical Support     high        536.0
1          2  Returns and Exchanges   medium        526.0
2          3   Billing and Payments      low        593.0

   ticket_id           tag
0          1       Account
1          1    Disruption
2          1        Outage
3          1            IT
4          1  Tech Support
5          2       Product
6          2       Feature
7          2  Tech Support

             tag      n
0   Tech Support  14152
1             IT  13993
2    Performance  11410
3       Feedback   8716
4  Documentation   7644
5            Bug   7532
6       Security   6596
7        Feature   5777
8     Disruption   4244
9         Outage   3593


## Results

The join worked: ticket 1 has five tag rows, ticket 2 has three.

There are 1,650 unique tags, but the distribution is heavily skewed. Tech Support appears 14,152 times while most tags appear only once or twice. Any analysis below needs a minimum frequency filter, or the results fill up with tags too rare to say anything about.

## 3. Ticket volume by department and priority

Checking how many tickets each department handles, and how the priority mix differs between them. Only working with the tickets table.

In [7]:
q("""
    SELECT
        department,
        COUNT(*) AS total,
        SUM(CASE WHEN priority = 'high'   THEN 1 ELSE 0 END) AS high,
        SUM(CASE WHEN priority = 'medium' THEN 1 ELSE 0 END) AS medium,
        SUM(CASE WHEN priority = 'low'    THEN 1 ELSE 0 END) AS low,
        ROUND(100.0 * SUM(CASE WHEN priority = 'high' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_high
    FROM tickets
    GROUP BY department
    ORDER BY total DESC
""")

,department,total,high,medium,low,pct_high
0,Technical Support,8617,5034,2590,993,58.4
1,Product Support,5539,1682,2809,1048,30.4
2,Customer Service,4482,828,2190,1464,18.5
3,IT Support,3500,1657,1496,347,47.3
4,Billing and Payments,3017,886,1499,632,29.4
5,Returns and Exchanges,1467,328,593,546,22.4
6,Service Outages and Maintenance,1157,818,196,143,70.7
7,Sales and Pre-Sales,885,154,408,323,17.4
8,Human Resources,568,59,238,271,10.4
9,General Inquiry,419,66,107,246,15.8


## Results

Priority mix varies sharply by department. Service Outages and
Maintenance is 70.7% high priority, against 10.4% for Human Resources.

That fits: an outage is urgent by definition, while an HR query usually
isn't. Technical Support handles both the largest volume (8,617) and a
high urgency rate (58.4%).

## 4. Most common tags

The first query that needs both tables. Counting tags means joining
ticket_tags back to tickets, which is only possible because the tags
were split out in the first place.

In [8]:
q("""
    SELECT
        t.tag,
        COUNT(*) AS tickets,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM tickets), 1) AS pct_of_tickets
    FROM ticket_tags t
    JOIN tickets tk ON tk.ticket_id = t.ticket_id
    GROUP BY t.tag
    HAVING COUNT(*) >= 100
    ORDER BY tickets DESC
    LIMIT 15
""")

,tag,tickets,pct_of_tickets
0,Tech Support,14152,47.7
1,IT,13993,47.2
2,Performance,11410,38.5
3,Feedback,8716,29.4
4,Documentation,7644,25.8
5,Bug,7532,25.4
6,Security,6596,22.2
7,Feature,5777,19.5
8,Disruption,4244,14.3
9,Outage,3593,12.1


## Results

Tech Support and IT each appear on nearly half of all tickets. Tags that
common behave more like defaults than descriptors.
The more specific ones further down, like Outage and Network, are the
ones likely to distinguish between departments.

HAVING is used here rather than WHERE because the filter applies after
grouping. It drops tags appearing fewer than 100 times, which is most
of the 1,650.

## 5. Top 3 tags per department

This needs ranking within groups, which a plain GROUP BY can't do.

The CTE counts how often each tag appears in each department. The window
function then ranks tags within each department separately. This is the function of
PARTITION BY, so the outer query can keep only the top three from
each.

In [9]:
q("""
    WITH tag_counts AS (
        SELECT
            tk.department,
            t.tag,
            COUNT(*) AS n
        FROM ticket_tags t
        JOIN tickets tk ON tk.ticket_id = t.ticket_id
        GROUP BY tk.department, t.tag
        HAVING COUNT(*) >= 20
    ),
    ranked AS (
        SELECT
            department,
            tag,
            n,
            RANK() OVER (PARTITION BY department ORDER BY n DESC) AS rank_in_dept
        FROM tag_counts
    )
    SELECT department, tag, n, rank_in_dept
    FROM ranked
    WHERE rank_in_dept <= 3
    ORDER BY department, rank_in_dept
""")

,department,tag,n,rank_in_dept
0,Billing and Payments,Billing,1681,1
1,Billing and Payments,Payment,1344,2
2,Billing and Payments,Tech Support,914,3
3,Customer Service,Tech Support,2134,1
4,Customer Service,IT,2110,2
5,Customer Service,Feedback,1735,3
6,General Inquiry,Feedback,213,1
7,General Inquiry,IT,155,2
8,General Inquiry,Performance,155,2
9,Human Resources,IT,297,1


## Results

Tech Support, IT and Performance take the top three places in seven of
the ten departments. Those tags appear on so many tickets that they say
almost nothing about which department a ticket belongs to.

Only three departments have a genuinely distinctive top tag: Billing and
Payments (Billing, Payment), Service Outages and Maintenance (Outage),
and Sales and Pre-Sales (Feedback).

General Inquiry shows a tie at rank 2, so RANK() assigns both the same rank and skips rank 3. That
is the difference between RANK() and ROW_NUMBER(), which would have
broken the tie arbitrarily.

## 6. Which tags actually distinguish a department

The previous query showed the most frequent tags are mostly noise. This
one asks the opposite question: which tags are concentrated in a single
department rather than spread across all of them?

For each tag, it finds the department where it appears most, then works
out what share of that tag's total usage falls there. A tag appearing
90% in one department is a strong signal. One spread evenly is not.

In [10]:
q("""
    WITH tag_dept AS (
        SELECT tk.department, t.tag, COUNT(*) AS n
        FROM ticket_tags t
        JOIN tickets tk ON tk.ticket_id = t.ticket_id
        GROUP BY tk.department, t.tag
    ),
    tag_total AS (
        SELECT tag, SUM(n) AS total
        FROM tag_dept
        GROUP BY tag
        HAVING SUM(n) >= 200
    )
    SELECT
        td.tag,
        td.department AS top_department,
        td.n AS in_dept,
        tt.total,
        ROUND(100.0 * td.n / tt.total, 1) AS pct_in_top_dept
    FROM tag_dept td
    JOIN tag_total tt ON tt.tag = td.tag
    WHERE td.n = (
        SELECT MAX(n) FROM tag_dept x WHERE x.tag = td.tag
    )
    ORDER BY pct_in_top_dept DESC
    LIMIT 15
""")

,tag,top_department,in_dept,total,pct_in_top_dept
0,Payment,Billing and Payments,1344,1354,99.3
1,Refund,Billing and Payments,440,452,97.3
2,Billing,Billing and Payments,1681,1738,96.7
3,Account,Billing and Payments,562,859,65.4
4,Service Disruption,Technical Support,172,305,56.4
5,Crash,Technical Support,880,1725,51.0
6,Urgent Issue,Technical Support,307,643,47.7
7,Software Bug,Technical Support,158,357,44.3
8,Virus,Technical Support,318,726,43.8
9,API,Technical Support,91,215,42.3


## Results

This is the clearest result so far. Payment, Refund and Billing appear
in Billing and Payments 96–99% of the time. Those three are near-perfect
predictors.

Nothing else comes close. Apart from Billing, no tag exceeds 65%
concentration, and most sit around 40%.

The tag "IT Support" is a good example: its most common department is
Technical Support at 40%, not IT Support. Even a tag named after a
department doesn't reliably indicate it, which is consistent with how
much those two departments overlap.

## 7. Which tags signal a high-priority ticket

Tags turned out to be weak at predicting department. This query
questions whether they do better at predicting urgency.

First, the overall rate, that is, the baseline any individual tag should be
compared against.

In [15]:
q("""
    SELECT
        COUNT(*) AS total_tickets,
        SUM(CASE WHEN priority = 'high' THEN 1 ELSE 0 END) AS high_priority,
        ROUND(100.0 * SUM(CASE WHEN priority = 'high' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_high
    FROM tickets
""")

,total_tickets,high_priority,pct_high
0,29651,11512,38.8


38.8% of all tickets are high priority. Any tag running well above that
is carrying real signal about severity.

In [16]:
q("""
    SELECT
        t.tag,
        COUNT(*) AS tickets,
        SUM(CASE WHEN tk.priority = 'high' THEN 1 ELSE 0 END) AS high_priority,
        ROUND(100.0 * SUM(CASE WHEN tk.priority = 'high' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_high
    FROM ticket_tags t
    JOIN tickets tk ON tk.ticket_id = t.ticket_id
    GROUP BY t.tag
    HAVING COUNT(*) >= 200
    ORDER BY pct_high DESC
    LIMIT 15
""")

,tag,tickets,high_priority,pct_high
0,Service Disruption,305,208,68.2
1,Urgent Issue,643,430,66.9
2,Crash,1725,1022,59.2
3,Outage,3593,1989,55.4
4,Recovery,1764,927,52.6
5,Server,228,119,52.2
6,IT Support,355,177,49.9
7,Maintenance,1092,542,49.6
8,Disruption,4244,2089,49.2
9,Service Recovery,254,121,47.6


## Results

Against a baseline of 38.8% overall, several tags
run well above it: Service Disruption at 68.2%, Urgent Issue at 66.9%,
Crash at 59.2%.

The whole top of the list is consistent: disruption,
crashes, outages, recovery.

So tags are weak at predicting which department a ticket belongs to, but
reasonably good at predicting how urgent it is. These are different
questions and the data answers them differently.

## 8. Department summary

A query to pull the analysis together, joining both tables to produce
a single view per department: volume, urgency, and its most
characteristic tag.

The CTE finds each department's top tag; the outer query joins that onto
the volume figures.

In [13]:
q("""
    WITH dept_top_tag AS (
        SELECT
            tk.department,
            t.tag,
            COUNT(*) AS n,
            ROW_NUMBER() OVER (PARTITION BY tk.department ORDER BY COUNT(*) DESC) AS rn
        FROM ticket_tags t
        JOIN tickets tk ON tk.ticket_id = t.ticket_id
        WHERE t.tag NOT IN ('Tech Support', 'IT', 'Performance')
        GROUP BY tk.department, t.tag
    )
    SELECT
        tk.department,
        COUNT(*) AS tickets,
        ROUND(100.0 * SUM(CASE WHEN tk.priority = 'high' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_high,
        ROUND(AVG(tk.body_length)) AS avg_length,
        dt.tag AS top_tag
    FROM tickets tk
    LEFT JOIN dept_top_tag dt
        ON dt.department = tk.department AND dt.rn = 1
    GROUP BY tk.department, dt.tag
    ORDER BY tickets DESC
""")

,department,tickets,pct_high,avg_length,top_tag
0,Technical Support,8617,58.4,394.0,Bug
1,Product Support,5539,30.4,380.0,Feedback
2,Customer Service,4482,18.5,385.0,Feedback
3,IT Support,3500,47.3,382.0,Bug
4,Billing and Payments,3017,29.4,385.0,Billing
5,Returns and Exchanges,1467,22.4,376.0,Feedback
6,Service Outages and Maintenance,1157,70.7,381.0,Outage
7,Sales and Pre-Sales,885,17.4,382.0,Feedback
8,Human Resources,568,10.4,352.0,Feedback
9,General Inquiry,419,15.8,398.0,Feedback


## Results

Two things stand out. Even after excluding Tech Support, IT and Performance, another generic
tag rises to replace them. Feedback is the top tag for six of the ten
departments. Only Billing and Payments and Service Outages and
Maintenance have a genuinely specific one.

Average body length is flat across every department, between 352 and
398 characters. Ticket length carries no information about where a
ticket belongs or how urgent it is.

## 9. Saving the queries

Writing the queries to a plain .sql file so they can be read without
opening the notebook.

In [17]:
queries = """
-- 1. Ticket volume and priority mix by department
SELECT department, COUNT(*) AS total,
       SUM(CASE WHEN priority='high' THEN 1 ELSE 0 END) AS high,
       ROUND(100.0*SUM(CASE WHEN priority='high' THEN 1 ELSE 0 END)/COUNT(*),1) AS pct_high
FROM tickets GROUP BY department ORDER BY total DESC;

-- 2. Most common tags (join + HAVING to drop the long tail)
SELECT t.tag, COUNT(*) AS tickets
FROM ticket_tags t JOIN tickets tk ON tk.ticket_id = t.ticket_id
GROUP BY t.tag HAVING COUNT(*) >= 100
ORDER BY tickets DESC LIMIT 15;

-- 3. Top 3 tags per department (CTE + RANK window function)
WITH tag_counts AS (
    SELECT tk.department, t.tag, COUNT(*) AS n
    FROM ticket_tags t JOIN tickets tk ON tk.ticket_id = t.ticket_id
    GROUP BY tk.department, t.tag HAVING COUNT(*) >= 20
), ranked AS (
    SELECT department, tag, n,
           RANK() OVER (PARTITION BY department ORDER BY n DESC) AS rank_in_dept
    FROM tag_counts
)
SELECT * FROM ranked WHERE rank_in_dept <= 3 ORDER BY department, rank_in_dept;

-- 4. Which tags concentrate in one department (correlated subquery)
WITH tag_dept AS (
    SELECT tk.department, t.tag, COUNT(*) AS n
    FROM ticket_tags t JOIN tickets tk ON tk.ticket_id = t.ticket_id
    GROUP BY tk.department, t.tag
), tag_total AS (
    SELECT tag, SUM(n) AS total FROM tag_dept GROUP BY tag HAVING SUM(n) >= 200
)
SELECT td.tag, td.department AS top_department,
       ROUND(100.0*td.n/tt.total,1) AS pct_in_top_dept
FROM tag_dept td JOIN tag_total tt ON tt.tag = td.tag
WHERE td.n = (SELECT MAX(n) FROM tag_dept x WHERE x.tag = td.tag)
ORDER BY pct_in_top_dept DESC LIMIT 15;

-- 5. Overall high-priority baseline
SELECT COUNT(*) AS total_tickets,
       ROUND(100.0*SUM(CASE WHEN priority='high' THEN 1 ELSE 0 END)/COUNT(*),1) AS pct_high
FROM tickets;

-- 6. Which tags signal high priority
SELECT t.tag, COUNT(*) AS tickets,
       ROUND(100.0*SUM(CASE WHEN tk.priority='high' THEN 1 ELSE 0 END)/COUNT(*),1) AS pct_high
FROM ticket_tags t JOIN tickets tk ON tk.ticket_id = t.ticket_id
GROUP BY t.tag HAVING COUNT(*) >= 200
ORDER BY pct_high DESC LIMIT 15;

-- 7. Department summary (ROW_NUMBER to pick one top tag each)
WITH dept_top_tag AS (
    SELECT tk.department, t.tag, COUNT(*) AS n,
           ROW_NUMBER() OVER (PARTITION BY tk.department ORDER BY COUNT(*) DESC) AS rn
    FROM ticket_tags t JOIN tickets tk ON tk.ticket_id = t.ticket_id
    WHERE t.tag NOT IN ('Tech Support','IT','Performance')
    GROUP BY tk.department, t.tag
)
SELECT tk.department, COUNT(*) AS tickets,
       ROUND(100.0*SUM(CASE WHEN tk.priority='high' THEN 1 ELSE 0 END)/COUNT(*),1) AS pct_high,
       ROUND(AVG(tk.body_length)) AS avg_length, dt.tag AS top_tag
FROM tickets tk
LEFT JOIN dept_top_tag dt ON dt.department = tk.department AND dt.rn = 1
GROUP BY tk.department, dt.tag ORDER BY tickets DESC;
"""

with open("queries.sql", "w") as f:
    f.write(queries.strip())

print("Saved queries.sql")

Saved queries.sql
